##### Setting Up

In [0]:
import pyspark.sql.functions as F 
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/tosinforlly@gmail.com/fmcg_project/1_setup/utilities

In [0]:
dbutils.widgets.text('catalog', 'fmcg', 'Catalog')
dbutils.widgets.text('data_source', 'customers', 'Data Source')

In [0]:
catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

storage_path = f's3://sport-bar/{data_source}/*.csv'

##### BRONZE LAYER

In [0]:
df = (
    spark.read.format("csv")
    .option("inferSchema", True)
    .option("header", True)
    .load(storage_path)
    .withColumn("ingest_timestamp", F.current_timestamp())
    .select("*", "_metadata.file_name")
)

In [0]:
df.write\
    .mode("overwrite")\
    .format("delta")\
    .saveAsTable(f'{catalog}.{bronze_schema}.{data_source}')

##### SILVER LAYER

In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source}")

display(df_bronze)

In [0]:
df_bronze.printSchema()

##### Transformation

- 1. Remove Duplicates

In [0]:
# check for duplicates in customer_id
dupl_chck = df_bronze.groupBy('customer_id').count().filter(F.col('count') > 1)

# Remove Duplicates
df_silver = df_bronze.dropDuplicates(['customer_id']).withColumn('customer_id', F.col('customer_id').cast('string'))


In [0]:
print(f'Number of rows before duplicates dropped: {df_bronze.count()}')

print(f'Number of rows after duplicates dropped: {df_silver.count()}')

- 2. Trim Whitespaces

In [0]:
# Trimming out white spaces from customer_name and city column
df_silver = df_silver\
    .withColumn('customer_name',
                F.trim(F.col('customer_name').alias('customer_name')))\
    .withColumn('city',
                F.trim(F.col('city').alias('city')))

In [0]:
# Quality Check
df_silver.filter( 
    (F.col('customer_name') != F.trim(F.col('customer_name'))) | 
    (F.col('city') != F.trim(F.col('city')))
    ).display()

In [0]:
%python
display(df_silver)


- 3. Correction of City names & INTCAP

In [0]:
# Scan for abnormalities
df_silver.select('city').distinct().display()

In [0]:
city_map = {
    'Bengaluruu': 'Bengaluru',
    'Bengalore': 'Bengaluru',

    'Hyderabadd': 'Hyderabad',
    'Hyderbad': 'Hyderabad',

    'NewDelhi': 'New Delhi',
    'NewDheli': 'New Delhi',
    'NewDelhee': 'New Delhi'
}

allowed_city = ['Bengaluru', 'Hyderabad', 'New Delhi']

df_silver = (
    df_silver.withColumn('customer_name', F.initcap(F.col('customer_name')))
    .replace(city_map, subset=["city"])
    .withColumn(
        "city",
        F.when(F.col("city").isNull(), None)
         .when(F.col("city").isin(allowed_city), F.col("city"))
         .otherwise(None)
    )
)

In [0]:
# Quality Check
df_silver.select('city').distinct().display()

- 4. Replacing Null in city

In [0]:
# Checking Nulls
df_silver.filter(F.col('city').isNull()).display()

In [0]:
# Customer citi Truth Supplied by Business Truth
city_truth = {
    # Sprintx Nutrition
    789403: "New Delhi",

    # Zenathlete Foods
    789420: "Bengaluru",

    # Primefuel Nutrition
    789521: "Hyderabad",

    # Recovery Lane
    789603: "Hyderabad"
}

# Make city_truth a dataframe
city_truth_df = spark.createDataFrame([(k,v) for k,v in city_truth.items()], ['customer_id', 'truth_city'])

display(city_truth_df)

In [0]:
display(df_silver)

In [0]:
# Join df_silver with city_truth_df, based on custormer_id
df_jn = df_silver.join(city_truth_df, 'customer_id', 'left')

# Coalesce column city to truth_city
df_coa = df_jn.withColumn('city', F.coalesce('city', 'truth_city'))

display(df_coa)



In [0]:
# Drop truth_city
df_silver = df_coa.drop('truth_city')
display(df_silver)

- 5. Standardization of Customer Data to Match Parent Data 


In [0]:
%sql
SELECT * FROM fmcg.gold.dim_customers;

In [0]:
df_silver = df_silver\
    .withColumn('market', F.lit('India'))\
    .withColumn('platform', F.lit('Sports Bar'))\
    .withColumn('channel', F.lit('Acquisition'))

In [0]:
df_silver = df_silver\
    .withColumn(
        "customer",
        F.concat_ws("-", "customer_name", F.coalesce(F.col("city"), F.lit("Unknown")))
    )\
    .withColumn("market", F.lit("India"))\
    .withColumn("platform", F.lit("Sports Bar"))\
    .withColumn("channel", F.lit("Acquisition"))

In [0]:
display(df_silver)

In [0]:

# Write into silver table
df_silver.write\
    .format('delta')\
    .mode('overwrite')\
    .option('mergeSchema', 'true')\
    .saveAsTable(f'{catalog}.{silver_schema}.{data_source}')

##### GOLD

In [0]:
df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source};")

In [0]:
display(df_silver)

In [0]:
df_gold = df_silver.select('customer_id', 'customer_name', 'customer', 'city', 'market', 'platform', 'channel')

In [0]:
display(df_gold)

In [0]:
# Write Into Gold Taable
df_gold.write\
.mode('overwrite')\
.format('delta')\
.option('delta.enableChangeDataFeed', 'true')\
.saveAsTable(f'{catalog}.{gold_schema}.sb_dim_{data_source}')

##### Merge Data With Parent table

In [0]:
delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_customers")
df_child_customers = spark.table("fmcg.gold.sb_dim_customers").select(
    F.col("customer_id").alias("customer_code"),
    "customer",
    "market",
    "platform",
    "channel"
)

In [0]:
delta_table.alias("target").merge(
    source=df_child_customers.alias("source"),
    condition="target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()